In [1]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00


In [2]:
from __future__ import annotations
import math
import numpy as np
from dataclasses import dataclass, field
from typing import Callable, Sequence
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import MCXGate
from qiskit.quantum_info import Statevector

In [ ]:
#Querying the QRAM

def fanout(data:Sequence[int]) -> QuantumCircuit:
    N = len(data)
    n = int(math.log2(N))
    assert 2**n == N, "Data length must be a power of two!"
    addr = QuantumRegister(n, name="addr") #Address
    out = QuantumRegister(1, name="data")
    qc = QuantumCircuit(addr, out, name="Fanout")
    for x, d in enumerate(data):
        if d == 0:
            continue
        bits = [(x >> k) & 1 for k in range(n)]
        for k, b in enumerate(bits):  #Flip the address bits that are 0 for operation
            if b == 0:
                qc.x(addr[k])
        qc.append(MCXGate(n), list(addr) + [out[0]])
        for k, b in enumerate(bits):
            if b == 0:
                qc.x(addr[k])
    return qc

def bucket_brigade(data:Sequence[int]) -> QuantumCircuit:
    N = len(data)
    n = int(math.log2(N))
    assert 2**n == N
    addr = QuantumRegister(n, name="addr")
    routers = QuantumRegister(2**n - 1, name="rtr")
    bus = QuantumRegister(1, name="bus")
    qc = QuantumCircuit(addr, routers, bus, name="BucketBrigade")

    def r_idx(L:int, i:int) -> int: #A forward sweep propagates address bits into the router qubits along the active path
        return (2**L - 1) + i       #At level L, the chosen router's state is copied from the L-th address qubit

    qc.cx(addr[0], routers[r_idx(0, 0)])

    for L in range(1, n): #Bucket brigade binary tree decomposes into a two control Toffoli
        for i in range(2**L):
            parent = r_idx(L-1, i//2)
            child = r_idx(L, i)
            if i%2 == 0:  #i is even(left child), parent must be 0 otherwise 1
                qc.x(routers[parent])
            qc.append(MCXGate(2), [addr[L], routers[parent], routers[child]])
            if i%2 == 0:
                qc.x(routers[parent])

    for k in range(N):  #Leaf k stores the data[k]
        if data[k] == 0:
            continue
        bits = [(k >> L) & 1 for L in range(n)]
        for L in range(n):  #Corresponds to that the address register is in |k>
            if bits[L] == 0:
                qc.x(addr[L])
        qc.append(MCXGate(n), list(addr) + [bus[0]])
        for L in range(n):
            if bits[L] == 0:
                qc.x(addr[L])

    for L in range(n-1, 0, -1): #A backward sweep frees the router qubits
        for i in range(2**L):
            parent = r_idx(L-1, i//2)
            child = r_idx(L, i)
            if i % 2 == 0:
                qc.x(routers[parent])
            qc.append(MCXGate(2), [addr[L], routers[parent], routers[child]])
            if i%2 == 0:
                qc.x(routers[parent])
    qc.cx(addr[0], routers[r_idx(0, 0)])

    return qc


In [ ]:
def batch_write_plan(data:Sequence[int]) -> list[tuple[int, int]]: #Returns a classical sequence of (address, value) pairs to write
    return [(i, v) for i, v in enumerate(data)]

In [ ]:
#Pre-query router and spin re-initialization

def refresh_circuit(addr:int) -> QuantumCircuit:
    routers = QuantumRegister(2**addr - 1, name="rtr")
    bus = QuantumRegister(1, name="bus")
    qc = QuantumCircuit(routers, bus, name="Refresh")
    for q in routers:
        qc.reset(q)
    qc.reset(bus[0])
    return qc

In [ ]:
def uncompute(circuit:QuantumCircuit) -> QuantumCircuit:
    return circuit.inverse()

In [ ]:
#Heralded retry

@dataclass
class HeraldedQueryResult:  #If the result is a heralded QRAM query, including retries
    success: bool
    attempts: int
    statevector_out: Statevector | None = None

def heralded_query(query_builder:Callable[[], QuantumCircuit], p_success:float, max_attempts:int = 50, rng:np.random.Generator | None = None) -> HeraldedQueryResult:
    rng = rng or np.random.default_rng()  #Simulates a heralded retry
    attempts = 0
    while attempts < max_attempts:
        attempts += 1
        if rng.random() < p_success:
            qc = query_builder()
            try:
                sv = Statevector.from_instruction(qc)
            except Exception:
                sv = None
            return HeraldedQueryResult(success=True, attempts=attempts, statevector_out=sv)
    return HeraldedQueryResult(success=False, attempts=attempts)

In [ ]:
#Hann-regime infidelity

def hann_infidel(eps:float, addr_bits:int, prefactor:float = 1.0) -> float:
    return prefactor*eps*(addr_bits**2)  #Upper bound on the query infidelity in Hann polylog regime

def effect_error(raw_err: float, erasure: float) -> float:
    return raw_err*(1.0 - erasure) #Reduce per router error by erasure conversion

In [ ]:
#Small demo

def analyse_qram(qram_fn:Callable[[Sequence[int]], QuantumCircuit], data:Sequence[int]) -> bool:
    N = len(data)
    n = int(math.log2(N))
    qc = qram_fn(data)
    for x in range(N):
        init = QuantumCircuit(*qc.qregs)  #Prepare |x> on the address register and |0> elsewhere
        bits = [(x >> k) & 1 for k in range(n)]
        for k, b in enumerate(bits):
            if b == 1:
                init.x(init.qregs[0][k])
        full = init.compose(qc)
        sv = Statevector.from_instruction(full)
        probs = sv.probabilities([full.num_qubits-1])
        expected = data[x]
        if not np.isclose(probs[expected], 1.0, atol=1e-10):
            print(f"Address Mismatch at {x}: expected bus={expected}," f"got probs={probs}")
            return False
    return True

def _demo_layer2() -> None:
    print("Layer 2")
    data = [1,0,1,1,0,1,0,0] #Fanout analyses example
    print(f"Test data D = {data} (N = {len(data)})")
    print("Fanout QRAM: ", end="")
    ok = analyse_qram(fanout, data)
    print("PASS" if ok else "FAIL")
    print("Bucket brigade QRAM: ", end="")
    ok = analyse_qram(bucket_brigade, data)
    print("PASS" if ok else "FAIL")

    fanout = fanout(data)
    bb = bucket_brigade(data)
    print(f"Fanout QRAM:{fanout.num_qubits:>3} qubits," f"depth {fanout.decompose().depth():>4}," f"size {fanout.decompose().size():>4}")
    print(f"Bucket brigade QRAM:{bb.num_qubits:>3} qubits," f"depth {bb.decompose().depth():>4}," f"size {bb.decompose().size():>4}")

    print("Hann polylog infidelity bound:")
    for n in (8,10,12,14,16,20):
        F = 1.0 - hann_infidel(1e-3, n)
        print(f"n = {n:>2} (N = 2^{n} = {2**n:>6})" f"1-F <= {hann_infidel(1e-3, n):.4f} F >= {F:.4f}")

    print("Effective error after erasure conversion:")
    for eff_err in (0.0,0.5,0.75,0.9,0.98):
        eps_eff = effect_error(1e-3, eff_err)
        print(f"Erasure = {eff_err:>4.2f} eps_eff = {eps_eff:.2e}")

    print("Heralded retry statistics:")
    rng = np.random.default_rng(0)
    data3 = [1,0,0,1]
    attempts = []
    fails = 0
    for _ in range(2000):
        r = heralded_query(lambda: fanout(data3), p_success=0.20, rng=rng)
        if r.success:
            attempts.append(r.attempts)
        else:
            fails += 1
    print(f"Mean attempts: {np.mean(attempts):.2f}")
    print(f"Theoretical 1/p: {1.0 / 0.20:.2f}")
    print(f"Failed trials: {fails}/2000")

In [ ]:
if __name__ == "__main__":
    _demo_layer2()